In [ ]:
# Enable CUDA GPU - SELECT GPU T4 ×2 ON KAGGLE!
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("🚀 CUDA GPU Setup...")
    !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy -q 2>/dev/null || true
    !pip install -q cupy-cuda11x
    
    try:
        import cupy as cp
        device = cp.cuda.Device()
        gpu_name = cp.cuda.runtime.getDeviceProperties(device.id)['name'].decode('utf-8')
        total_mem, free_mem = cp.cuda.Device().mem_info
        print(f"✓ GPU: {gpu_name}, Memory: {total_mem / 1e9:.1f} GB")
        USE_GPU = True
    except Exception as e:
        print(f"⚠️ GPU setup failed: {e}")
        USE_GPU = False
else:
    print("Local CPU mode")
    USE_GPU = False

print("="*60)

In [ ]:
# Clone Repository
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    if not os.path.exists('/kaggle/working/NLP_PROJECT_2025'):
        !git clone -q -b mohab https://github.com/MohabYasser2/NLP_PROJECT_2025.git /kaggle/working/NLP_PROJECT_2025
    else:
        !cd /kaggle/working/NLP_PROJECT_2025 && git fetch -q origin mohab && git reset --hard origin/mohab
    
    # Clear old cache
    !rm -f /kaggle/working/*_processed.pkl
    sys.path.insert(0, '/kaggle/working/NLP_PROJECT_2025')
    print("✓ Repo ready")
else:
    sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# Detect Dataset Paths
import glob

if IS_KAGGLE:
    data_files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    if data_files:
        dataset_dir = os.path.dirname(data_files[0])
        TRAIN_FILE = os.path.join(dataset_dir, 'train.txt')
        DEV_FILE = os.path.join(dataset_dir, 'val.txt') if 'val.txt' in str(data_files) else os.path.join(dataset_dir, 'dev.txt')
        TEST_FILE = os.path.join(dataset_dir, 'test.txt')
    else:
        print("⚠️ No dataset found! Add your dataset in Kaggle.")
else:
    TRAIN_FILE = '../data/train.txt'
    DEV_FILE = '../data/val.txt'
    TEST_FILE = '../data/test.txt'

OUTPUT_FILE = '/kaggle/working/submission.csv' if IS_KAGGLE else 'submission.csv'
print(f"✓ Train: {TRAIN_FILE}")

In [ ]:
# MEMORY-OPTIMIZED TRAINING FOR GPU T4 x2 (30GB RAM LIMIT)
import pickle
import numpy as np
from pathlib import Path
from src.preprocessing import prepare_dataset
from src.models.logreg_model import LogisticRegressionModel

MODEL_PATH = '/kaggle/working/NLP_PROJECT_2025/models/logreg_model.pkl' if IS_KAGGLE else '../models/logreg_model.pkl'

if os.path.exists(MODEL_PATH):
    print("="*70)
    print("📦 LOADING PRE-TRAINED MODEL")
    print("="*70)
    
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    
    # Load dev data
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    cache_file = cache_dir / 'val_processed.pkl'
    
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(cache_file.with_suffix('')))
    
    # Evaluate
    metrics = model.evaluate(dev_texts, dev_labels, window_size=7)
    print(f"\n✓ Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"✓ DER: {metrics['der']*100:.2f}%")
    print("="*70)
    
else:
    print("="*70)
    print("🚀 ULTRA-OPTIMIZED TRAINING (GPU T4 x2, <30GB RAM)")
    print("="*70)
    print("\n📋 Memory-Optimized Configuration:")
    print("   • Features: 5,000 (reduced from 10k to save RAM)")
    print("   • N-grams: 1-3 (reduced from 1-4)")
    print("   • Batch processing: 10k samples at a time")
    print("   • Vocab sampling: 20k sentences (not full 50k)")
    print("   • Window: 7, LR: 0.05, Iter: 150, Batch: 512")
    print("\n⏱️  Expected: ~15-20 min, Peak RAM: ~18-22GB")
    print("="*70)
    
    # Load data with chunking
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    
    print("\n[1/5] Loading training data...")
    train_cache = cache_dir / 'train_processed.pkl'
    if train_cache.exists():
        with open(train_cache, 'rb') as f:
            data = pickle.load(f)
        train_texts, train_labels = data['texts'], data['labels']
    else:
        train_texts, train_labels = prepare_dataset(str(TRAIN_FILE), str(train_cache.with_suffix('')))
    
    print(f"✓ Loaded {len(train_texts)} training samples")
    
    print("\n[2/5] Loading dev data...")
    dev_cache = cache_dir / 'val_processed.pkl'
    if dev_cache.exists():
        with open(dev_cache, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(dev_cache.with_suffix('')))
    
    print(f"✓ Loaded {len(dev_texts)} dev samples")
    
    # CRITICAL: Reduce features and use vocabulary sampling
    print("\n[3/5] Building model with memory-optimized params...")
    model = LogisticRegressionModel(
        learning_rate=0.05,
        max_iter=150,
        regularization=5e-4,
        max_features=5000,  # REDUCED: 5k instead of 10k
        ngram_range=(1, 3),  # REDUCED: 1-3 instead of 1-4
        batch_size=512  # LARGER: process more at once
    )
    
    # Extract contexts with chunking
    print("\n[4/5] Extracting contexts (memory-efficient)...")
    window_size = 7
    
    # Process in chunks to avoid memory spike
    chunk_size = 10000
    train_contexts_list = []
    train_context_labels_list = []
    
    for i in range(0, len(train_texts), chunk_size):
        chunk_texts = train_texts[i:i+chunk_size]
        chunk_labels = train_labels[i:i+chunk_size]
        
        chunk_contexts = []
        chunk_context_labels = []
        
        for text, label_seq in zip(chunk_texts, chunk_labels):
            # Ensure text and labels have same length
            min_len = min(len(text), len(label_seq))
            for j in range(min_len):
                start = max(0, j - window_size // 2)
                end = min(len(text), j + window_size // 2 + 1)
                context = text[start:end]
                chunk_contexts.append(context)
                chunk_context_labels.append(label_seq[j])
        
        train_contexts_list.extend(chunk_contexts)
        train_context_labels_list.extend(chunk_context_labels)
        print(f"  Processed {i+len(chunk_texts)}/{len(train_texts)} sentences", end='\r')
    
    print(f"\n✓ Generated {len(train_contexts_list):,} training contexts")
    
    # Dev contexts (smaller, can do at once)
    dev_contexts = []
    dev_context_labels = []
    for text, label_seq in zip(dev_texts, dev_labels):
        # Ensure text and labels have same length
        min_len = min(len(text), len(label_seq))
        for j in range(min_len):
            start = max(0, j - window_size // 2)
            end = min(len(text), j + window_size // 2 + 1)
            context = text[start:end]
            dev_contexts.append(context)
            dev_context_labels.append(label_seq[j])
    
    print(f"✓ Generated {len(dev_contexts):,} dev contexts")
    
    # CRITICAL: Sample vocabulary from subset (not full data)
    print("\n[5/5] Training model (GPU-accelerated)...")
    print("📊 Using 20k sample for vocabulary (saves RAM)...")
    
    # Use only 20k samples for vocabulary
    vocab_sample = train_contexts_list[:20000]
    model.vectorizer.fit(vocab_sample)
    print(f"✓ Vocabulary built: {len(model.vectorizer.vocabulary):,} features")
    
    # Transform in mini-batches
    print("🔄 Transforming training data (batched)...")
    transform_batch_size = 50000
    X_train_batches = []
    
    for i in range(0, len(train_contexts_list), transform_batch_size):
        batch = train_contexts_list[i:i+transform_batch_size]
        X_batch = model.vectorizer.transform(batch, batch_size=transform_batch_size)
        X_train_batches.append(X_batch)
        print(f"  Transformed {i+len(batch):,}/{len(train_contexts_list):,}", end='\r')
    
    print()
    X_train = np.vstack(X_train_batches)
    del X_train_batches  # Free memory
    print(f"✓ Training matrix: {X_train.shape}, {X_train.nbytes / 1e9:.2f} GB")
    
    print("🔄 Transforming dev data...")
    X_dev = model.vectorizer.transform(dev_contexts, batch_size=50000)
    print(f"✓ Dev matrix: {X_dev.shape}, {X_dev.nbytes / 1e9:.2f} GB")
    
    # Train on GPU
    print("\n🚀 Training on GPU...")
    model.fit(X_train, train_context_labels_list)
    
    # Evaluate
    print("\n📊 Evaluating...")
    metrics = model.evaluate(dev_texts, dev_labels, window_size=7)
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE")
    print("="*70)
    print(f"✓ Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"✓ DER: {metrics['der']*100:.2f}%")
    print(f"✓ Correct: {metrics['correct']:,} / {metrics['total']:,}")
    print("="*70)
    
    # Save model
    Path(MODEL_PATH).parent.mkdir(exist_ok=True, parents=True)
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Model saved: {MODEL_PATH}")
